# Donor Lapse/Churn Intelligence: Snowflake ML → Agent

This hands-on lab builds the **complete Snowflake ML lifecycle** for donor lapse/churn prediction — for a generic **nonprofit fundraising CRM** — and finishes with a **Cortex Agent that calls the deployed model as a tool**.

### What you'll build
1. **Feature Store** — donor entity + RFM / engagement / tenure / wealth Feature Views (point-in-time correct)
2. **Datasets** — an immutable, versioned training set
3. **Cortex ML Functions** — Forecast, Anomaly Detection, Classification (no-code baseline)
4. **Snowpark ML** — an XGBoost lapse classifier + distributed HPO
5. **ML Jobs / Container Runtime** — scalable remote training (GPU optional)
6. **Model Registry** — versioning, metrics, DEFAULT promotion
7. **Explainability** — Shapley risk drivers per donor
8. **Model Serving** — batch + single-donor scoring
9. **ML Observability** — drift / performance monitor + alert
10. **Tool wrappers + Cortex Agent** — model-as-a-tool, planned retrieve → score → explain

**Prerequisite:** run `lab/setup.sql` first. Requires `snowflake-ml-python >= 1.26`.

---
## Section 1: Connect & Explore

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE DONOR_CHURN_ML_DEMO").collect()
session.sql("USE SCHEMA RAW").collect()
session.sql("USE WAREHOUSE DONOR_CHURN_ML_WH").collect()
print("Connected to DONOR_CHURN_ML_DEMO")

import snowflake.ml
print("snowflake-ml-python:", snowflake.ml.version.VERSION)

In [ ]:
session.sql("""
    SELECT is_lapsed,
           COUNT(*)                         AS donors,
           ROUND(AVG(recency_days))         AS avg_recency_days,
           ROUND(AVG(frequency_last_12m),2) AS avg_gifts_12m,
           ROUND(AVG(engagement_last_6m),2) AS avg_engagement_6m,
           ROUND(AVG(monetary_total))       AS avg_lifetime_giving
    FROM DONOR_TRAINING_BASE
    GROUP BY is_lapsed
    ORDER BY is_lapsed
""").show()

---
## Section 2: Feature Store

Register a `donor` **Entity** and **Feature Views** for RFM, engagement, and wealth signals. Feature Views are retrieved **point-in-time correct**, so training and inference use identical definitions (no train/serve skew).

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database="DONOR_CHURN_ML_DEMO",
    name="FEATURES",
    default_warehouse="DONOR_CHURN_ML_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXISTS,
)

donor = Entity(name="DONOR", join_keys=["DONOR_ID"], desc="A donor of the nonprofit fundraising CRM")
fs.register_entity(donor)
print("Entities:", [e.name for e in fs.list_entities().collect()])

In [ ]:
feature_df = session.sql("""
    SELECT donor_id,
           DATEADD('month', -12, CURRENT_DATE())::TIMESTAMP_NTZ AS as_of_ts,
           wealth_signal_score, tenure_days,
           frequency_lifetime, frequency_last_12m, monetary_total, avg_gift_amount,
           recency_days, engagement_count, engagement_last_6m
    FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE
""")

donor_rfm_fv = FeatureView(
    name="DONOR_RFM_FEATURES",
    entities=[donor],
    feature_df=feature_df,
    timestamp_col="AS_OF_TS",
    desc="Point-in-time RFM, engagement, tenure, and wealth features per donor",
)
donor_rfm_fv = fs.register_feature_view(feature_view=donor_rfm_fv, version="v1", overwrite=True)
print("Registered feature view DONOR_RFM_FEATURES/v1")

---
## Section 3: Datasets

Materialize an **immutable, versioned training set** from the Feature Store using a spine (donor_id + as-of date + label). Point-in-time joins guarantee no label leakage.

In [ ]:
spine_df = session.sql("""
    SELECT donor_id,
           DATEADD('month', -12, CURRENT_DATE())::TIMESTAMP_NTZ AS as_of_ts,
           is_lapsed
    FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE
""")

training_dataset = fs.generate_dataset(
    name="DONOR_CHURN_ML_DEMO.FEATURES.LAPSE_TRAINING_SET",
    version="v1",
    spine_df=spine_df,
    features=[donor_rfm_fv],
    spine_timestamp_col="AS_OF_TS",
    spine_label_cols=["IS_LAPSED"],
    desc="Point-in-time donor lapse training set",
)
print("Materialized dataset LAPSE_TRAINING_SET/v1")
train_sp_df = training_dataset.read.to_snowpark_dataframe()
train_sp_df.limit(5).show()

---
## Section 4: Cortex ML Functions (no-code bridge)

`setup.sql` created three built-in **Cortex ML Function** objects. Here we exercise them: **Forecast**, **Anomaly Detection**, and **Classification** (baseline lapse model + feature importance).

In [ ]:
# 4a. FORECAST — next 13 weeks (~1 quarter) of donation volume.
session.sql("USE SCHEMA MODELS").collect()
session.sql("USE WAREHOUSE DONOR_CHURN_ML_SOWH").collect()
session.sql("CALL DONATION_VOLUME_FORECAST!FORECAST(FORECASTING_PERIODS => 13)").show()

In [ ]:
# 4b. ANOMALY DETECTION — flag unusual weeks of giving drop-off / spikes.
session.sql("""
    SELECT * FROM TABLE(DONATION_VOLUME_ANOMALY!DETECT_ANOMALIES(
        INPUT_DATA => TABLE(DONOR_CHURN_ML_DEMO.RAW.DONATION_VOLUME_TS),
        TIMESTAMP_COLNAME => 'WEEK_TS',
        TARGET_COLNAME => 'TOTAL_AMOUNT'))
    WHERE is_anomaly = TRUE
    ORDER BY ts
""").show()

In [ ]:
# 4c. CLASSIFICATION — baseline lapse model: evaluation + feature importance.
session.sql("CALL LAPSE_BASELINE_MODEL!SHOW_EVALUATION_METRICS()").show()
session.sql("CALL LAPSE_BASELINE_MODEL!SHOW_FEATURE_IMPORTANCE()").show()
session.sql("USE WAREHOUSE DONOR_CHURN_ML_WH").collect()

---
## Section 5: Snowpark ML Modeling + Distributed HPO

Train a custom **XGBoost** lapse classifier with the `snowflake.ml.modeling` API — the fit runs in Snowflake, no data leaves. Then tune hyperparameters with distributed search.

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import roc_auc_score, f1_score

FEATURES = [
    "WEALTH_SIGNAL_SCORE", "TENURE_DAYS", "FREQUENCY_LIFETIME", "FREQUENCY_LAST_12M",
    "MONETARY_TOTAL", "AVG_GIFT_AMOUNT", "RECENCY_DAYS", "ENGAGEMENT_COUNT", "ENGAGEMENT_LAST_6M",
]
LABEL = "IS_LAPSED"

train_df, val_df = train_sp_df.random_split([0.8, 0.2], seed=42)

model = XGBClassifier(
    input_cols=FEATURES, label_cols=[LABEL], output_cols=["PRED_LAPSE"],
    n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
)
model.fit(train_df)

preds = model.predict(val_df)
auc = roc_auc_score(df=preds, y_true_col_names=LABEL, y_score_col_names="PRED_LAPSE")
f1 = f1_score(df=preds, y_true_col_names=LABEL, y_pred_col_names="PRED_LAPSE")
print(f"Validation AUC={auc:.3f}  F1={f1:.3f}")

In [ ]:
# Distributed hyperparameter optimization over a small grid (fits run across the warehouse).
from snowflake.ml.modeling.model_selection import GridSearchCV
from xgboost import XGBClassifier as SkXGB

grid = GridSearchCV(
    estimator=SkXGB(eval_metric="logloss"),
    param_grid={"n_estimators": [200, 400], "max_depth": [4, 6], "learning_rate": [0.05, 0.1]},
    scoring="f1",
    input_cols=FEATURES, label_cols=[LABEL], output_cols=["PRED_LAPSE"],
)
grid.fit(train_df)
print("Best params:", grid.to_sklearn().best_params_)
best_model = grid

---
## Section 6: ML Jobs on Container Runtime

Productionize training as a **remote ML Job** on a Snowflake **compute pool** (Container Runtime). The `@remote` decorator ships the function to Snowflake compute; add a GPU instance family for deep models. Requires a compute pool.

In [ ]:
# Create the compute pool if your role has the privilege.
try:
    session.sql("""
        CREATE COMPUTE POOL IF NOT EXISTS DONOR_CHURN_ML_POOL
          MIN_NODES = 1 MAX_NODES = 2 INSTANCE_FAMILY = CPU_X64_M
    """).collect()
    print("Compute pool ready: DONOR_CHURN_ML_POOL")
except Exception as e:
    print("Skipping compute-pool creation (needs CREATE COMPUTE POOL privilege):", e)

session.sql("CREATE STAGE IF NOT EXISTS DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE").collect()

In [ ]:
from snowflake.ml.jobs import remote

@remote("DONOR_CHURN_ML_POOL", stage_name="DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE", session=session)
def train_lapse_model(dataset_name: str, dataset_version: str):
    # Runs remotely on Container Runtime. Any OSS package is available.
    from snowflake.snowpark.context import get_active_session
    from snowflake.ml.dataset import load_dataset
    from snowflake.ml.modeling.xgboost import XGBClassifier
    s = get_active_session()
    ds = load_dataset(s, dataset_name, dataset_version)
    df = ds.read.to_snowpark_dataframe()
    feats = ["WEALTH_SIGNAL_SCORE","TENURE_DAYS","FREQUENCY_LIFETIME","FREQUENCY_LAST_12M",
             "MONETARY_TOTAL","AVG_GIFT_AMOUNT","RECENCY_DAYS","ENGAGEMENT_COUNT","ENGAGEMENT_LAST_6M"]
    m = XGBClassifier(input_cols=feats, label_cols=["IS_LAPSED"], output_cols=["PRED_LAPSE"],
                      n_estimators=400, max_depth=6, learning_rate=0.05)
    m.fit(df)
    return m

try:
    job = train_lapse_model("DONOR_CHURN_ML_DEMO.FEATURES.LAPSE_TRAINING_SET", "v1")
    print("Job:", job.id, "status:", job.status)
    remote_model = job.result()
    print("Remote training complete.")
except Exception as e:
    print("ML Job needs a compute pool + snowflake-ml-python>=1.26. Using the local model instead.", e)
    remote_model = None

---
## Section 7: Model Registry

Log the trained model as a **versioned, governed object** with metrics, signature, and sample input. Target WAREHOUSE and SPCS serving, then promote to **DEFAULT**.

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="DONOR_CHURN_ML_DEMO", schema_name="MODELS")

mv = reg.log_model(
    model=best_model,
    model_name="DONOR_LAPSE_MODEL",
    version_name="v1",
    metrics={"auc": float(auc), "f1": float(f1)},
    sample_input_data=train_df.select(FEATURES).limit(100),
    comment="XGBoost donor lapse classifier (RFM + engagement + wealth features)",
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
)
print("Logged DONOR_LAPSE_MODEL/v1")
mv.show_functions()

reg.get_model("DONOR_LAPSE_MODEL").default = "v1"
print("DEFAULT version set to v1")

---
## Section 8: Model Explainability

Compute **Shapley attributions** so fundraisers see *why* a donor is at risk. Explainability runs on the registered model — no separate tooling.

In [ ]:
explain_input = val_df.select(FEATURES).limit(10)
try:
    shap_df = mv.run(explain_input, function_name="explain")
    shap_df.show()
except Exception as e:
    print("If `explain` is not registered, set enable_explainability=True at log_model time.", e)

---
## Section 9: Model Serving

Score the whole donor base in **batch**, and a **single donor** in real time — both from the same registered model.

In [ ]:
all_features = session.sql(f"""
    SELECT donor_id, {', '.join(FEATURES)}
    FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE
""")
scored = mv.run(all_features, function_name="predict")
scored.write.mode("overwrite").save_as_table("DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES")
session.sql("""
    SELECT donor_id, PRED_LAPSE
    FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES
    ORDER BY PRED_LAPSE DESC LIMIT 10
""").show()

In [ ]:
one_donor = session.sql(f"""
    SELECT donor_id, {', '.join(FEATURES)}
    FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE
    WHERE donor_id = 40821
""")
mv.run(one_donor, function_name="predict").show()

# Large-scale / GPU batch inference alternative:
#   from snowflake.ml.model.batch import OutputSpec
#   mv.run_batch(X=all_features, compute_pool="DONOR_CHURN_ML_POOL",
#                output_spec=OutputSpec(stage_location="@DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE/scores/"))

---
## Section 10: ML Observability

Create a **model monitor** tracking prediction drift, data drift, and accuracy over time — segmented by region — and query the metrics. Constraints: `TIMESTAMP_NTZ` timestamp, `NUMBER` prediction/actual columns, monitor in the same schema as the model, no NULLs/NaNs.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG AS
    SELECT b.donor_id, b.region,
           CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS scored_at,
           b.wealth_signal_score, b.tenure_days, b.frequency_lifetime,
           b.frequency_last_12m, b.monetary_total, b.avg_gift_amount,
           b.recency_days, b.engagement_count, b.engagement_last_6m,
           s.PRED_LAPSE::NUMBER(10,6) AS pred_lapse,
           b.is_lapsed::NUMBER        AS actual_lapse
    FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b
    JOIN DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s USING (donor_id)
""").collect()
session.sql("""
    CREATE OR REPLACE TABLE DONOR_CHURN_ML_DEMO.MODELS.LAPSE_BASELINE_SNAPSHOT AS
    SELECT * FROM DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG
""").collect()
print("Scoring log + baseline snapshot ready.")

In [ ]:
session.sql("""
    CREATE OR REPLACE MODEL MONITOR DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_MONITOR WITH
        MODEL = DONOR_LAPSE_MODEL
        VERSION = 'v1'
        FUNCTION = 'predict'
        SOURCE = DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG
        BASELINE = DONOR_CHURN_ML_DEMO.MODELS.LAPSE_BASELINE_SNAPSHOT
        WAREHOUSE = DONOR_CHURN_ML_WH
        REFRESH_INTERVAL = '1 day'
        AGGREGATION_WINDOW = '1 day'
        TIMESTAMP_COLUMN = scored_at
        ID_COLUMNS = ('DONOR_ID')
        PREDICTION_SCORE_COLUMNS = ('PRED_LAPSE')
        ACTUAL_CLASS_COLUMNS = ('ACTUAL_LAPSE')
        SEGMENT_COLUMNS = ('REGION')
""").collect()
print("Model monitor created. Query drift with MODEL_MONITOR_DRIFT_METRIC:")
session.sql("""
    SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        'DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_MONITOR',
        'DIFFERENCE_OF_MEANS', 'PRED_LAPSE', '1 DAY',
        DATEADD('day', -30, CURRENT_DATE()), CURRENT_DATE()))
""").show()

In [ ]:
# Reference alert (needs an email notification integration). Adjust names before enabling.
print("""
CREATE OR REPLACE ALERT DONOR_CHURN_ML_DEMO.MODELS.LAPSE_MODEL_ACCURACY_ALERT
  WAREHOUSE = DONOR_CHURN_ML_WH
  SCHEDULE = '1440 MINUTE'
  IF (EXISTS (
      SELECT 1 FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
          'DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_MONITOR',
          'F1', '1 DAY', DATEADD('day', -1, CURRENT_DATE()), CURRENT_DATE()))
      WHERE METRIC_VALUE < 0.80))
  THEN CALL SYSTEM$SEND_EMAIL('my_email_int','team@example.org',
       'Donor lapse model F1 dropped below 0.80', 'Investigate drift in the monitor dashboard.');
""")

---
## Section 11: Tool Wrappers (model as a callable tool)

Wrap the deployed model in SQL so an agent can call it: `TOP_CHURN_RISK(segment, n)` ranks the highest-risk donors in a segment; `PREDICT_DONOR_CHURN(donor_id)` scores one donor. Each returns churn probability, top risk drivers, and (for the batch variant) a recommended action via `AI_COMPLETE`.

In [ ]:
session.sql("""
    CREATE OR REPLACE FUNCTION DONOR_CHURN_ML_DEMO.MODELS.TOP_CHURN_RISK(SEGMENT STRING, N NUMBER)
    RETURNS TABLE (donor_id NUMBER, region STRING, churn_probability FLOAT,
                   risk_drivers STRING, recommended_action STRING)
    AS
    $$
        SELECT s.donor_id, b.region,
               ROUND(s.PRED_LAPSE, 4) AS churn_probability,
               'recency ' || b.recency_days || 'd; gifts_12m ' || b.frequency_last_12m
                 || '; engagement_6m ' || b.engagement_last_6m AS risk_drivers,
               SNOWFLAKE.CORTEX.AI_COMPLETE('llama3.1-8b',
                 'A ' || b.donor_segment || ' donor in ' || b.region
                 || ' has churn probability ' || ROUND(s.PRED_LAPSE,2)
                 || '. Last gift ' || b.recency_days || ' days ago, ' || b.frequency_last_12m
                 || ' gifts in the last year, ' || b.engagement_last_6m
                 || ' engagements in 6 months. Recommend one concise next-best retention action.'
               ) AS recommended_action
        FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s
        JOIN DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b USING (donor_id)
        WHERE b.donor_segment = SEGMENT
        ORDER BY s.PRED_LAPSE DESC
        LIMIT N
    $$;
""").collect()
print("Created TOP_CHURN_RISK. Example (top 5 Major donors):")
session.sql("SELECT * FROM TABLE(DONOR_CHURN_ML_DEMO.MODELS.TOP_CHURN_RISK('Major', 5))").show()

In [ ]:
session.sql("""
    CREATE OR REPLACE FUNCTION DONOR_CHURN_ML_DEMO.MODELS.PREDICT_DONOR_CHURN(DONOR_ID NUMBER)
    RETURNS TABLE (donor_id NUMBER, churn_probability FLOAT, risk_drivers STRING)
    AS
    $$
        SELECT s.donor_id, ROUND(s.PRED_LAPSE, 4) AS churn_probability,
               'recency ' || b.recency_days || 'd; gifts_12m ' || b.frequency_last_12m
                 || '; engagement_6m ' || b.engagement_last_6m AS risk_drivers
        FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s
        JOIN DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b USING (donor_id)
        WHERE s.donor_id = PREDICT_DONOR_CHURN.DONOR_ID
    $$;
""").collect()
session.sql("SELECT * FROM TABLE(DONOR_CHURN_ML_DEMO.MODELS.PREDICT_DONOR_CHURN(40821))").show()

---
## Section 12: Cortex Agent (model as a tool)

Register an agent with **two tools**: Cortex Analyst (retrieval over the semantic view) and a **custom tool** backed by `TOP_CHURN_RISK` (model scoring + explanation). The agent plans: retrieve a segment via Analyst → score via the model tool → return an explained, ranked action list.

> The `type: generic` custom tool is most reliably wired through **Snowsight → AI & ML → Agents → Custom tools** (select the `TOP_CHURN_RISK` function + a warehouse). The `CREATE AGENT` below sets up the Analyst tool and declares the custom tool; finish the binding in the UI, then grant USAGE on the function to the agent's role.

In [ ]:
session.sql("""
    CREATE OR REPLACE AGENT DONOR_CHURN_ML_DEMO.MODELS.DONOR_RETENTION_AGENT
      COMMENT = 'Donor retention agent: Analyst retrieval + deployed lapse model scoring'
      PROFILE = '{"display_name": "Donor Retention Intelligence", "color": "blue"}'
      FROM SPECIFICATION
      $$
      models:
        orchestration: auto
      instructions:
        response: "Be concise. When ranking donors, show churn probability, the top risk drivers, and a recommended retention action."
        orchestration: "Use Donor_Metrics (Cortex Analyst) to retrieve or aggregate donors by region, segment, giving, or lapse rate. Use Score_Donor_Churn to score and explain a set of donors with the deployed model. For 'who is most at risk and what should we do', first retrieve the segment with Analyst, then score with Score_Donor_Churn, then rank and explain."
        sample_questions:
          - question: "Which major-gift donors in the West region are most at risk of lapsing this quarter, and what should we do?"
          - question: "What is the lapse rate by region?"
      tools:
        - tool_spec:
            type: "cortex_analyst_text_to_sql"
            name: "Donor_Metrics"
            description: "Donor, donation, engagement, and lapse-rate metrics from the governed semantic view."
        - tool_spec:
            type: "generic"
            name: "Score_Donor_Churn"
            description: "Scores the highest-risk donors in a segment with the deployed lapse model; returns churn probability, risk drivers, and a recommended action."
      tool_resources:
        Donor_Metrics:
          semantic_view: "DONOR_CHURN_ML_DEMO.ANALYTICS.DONOR_CHURN_SV"
          execution_environment:
            type: "warehouse"
            warehouse: "DONOR_CHURN_ML_WH"
      $$
""").collect()
print("Agent DONOR_RETENTION_AGENT created.")
print("Finish binding Score_Donor_Churn -> TOP_CHURN_RISK in Snowsight (AI & ML -> Agents -> Custom tools),")
print("then grant USAGE on the function to the agent's role.")

---
## Section 13: The "Wow" Moment & Summary

Chat with the agent (Snowsight **AI & ML → Agents**, or `app/streamlit_app.py`):

1. *"Which of our major-gift donors in the West region are most at risk of lapsing this quarter, why, and what should we do?"* — retrieve via Analyst, score via the model tool, return a ranked, explained action list.
2. *"Draft outreach for the top 3."* — personalized stewardship messages grounded in each donor's risk drivers.

### What you built

| Capability | Object |
|---|---|
| Feature Store | `FEATURES` — `DONOR` entity, `DONOR_RFM_FEATURES/v1` |
| Dataset | `LAPSE_TRAINING_SET/v1` |
| Cortex ML Functions | `DONATION_VOLUME_FORECAST`, `DONATION_VOLUME_ANOMALY`, `LAPSE_BASELINE_MODEL` |
| Snowpark ML + HPO | `XGBClassifier` + `GridSearchCV` |
| ML Jobs | `@remote` training on `DONOR_CHURN_ML_POOL` |
| Registry | `DONOR_LAPSE_MODEL/v1` (DEFAULT) |
| Explainability | Shapley via `mv.run(function_name='explain')` |
| Serving | `DONOR_LAPSE_SCORES` (batch) + single-donor |
| Observability | `DONOR_LAPSE_MONITOR` + metric functions + alert |
| Tool wrappers | `TOP_CHURN_RISK`, `PREDICT_DONOR_CHURN` |
| Agent | `DONOR_RETENTION_AGENT` (Analyst + model tool) |

Everything ran inside one governed Snowflake account — no data movement.